In [1]:
from src.runtime_constants import rr
from src.server.models.gpt_tagger.prompts_helper import PromptsDef
from src.server.models.gpt_tagger.prompt_handlers_v2 import PromptHandlerTags, PromptHandlerPriorityTags
import json

from src.db.mysql import DbConnection

#rr.set_env(rr.ENV_TEST)
#conn = DbConnection(rr.ENV_TEST, is_remote=True)
conn = DbConnection()
rows = conn.select_all("""
SELECT p.id, p.result, p.prompt_id
  FROM tagged_data.gpt_prompts p
 WHERE p.prompt_id IN ('gpt4_genres_topics_prio')
 order by p.id desc
 LIMIT 10
""")

prompts_helper = PromptsDef(conn=conn)

for row in rows:
    result = json.loads(row["result"])
    parsed = PromptHandlerPriorityTags.parse(
        result=result,
        prompts_helper=prompts_helper
    )

    print(row["id"])
    print(parsed)

conn.close()

866
{'state': 'done', 'tags_details': [{'tag_id_int': 28, 'tag_rank': 1}, {'tag_id_int': 27, 'tag_rank': 2}, {'tag_id_int': 37, 'tag_rank': 3}, {'tag_id_int': 231, 'tag_rank': 1}, {'tag_id_int': 234, 'tag_rank': 2}, {'tag_id_int': 238, 'tag_rank': 3}], 'unknown': []}
860
{'state': 'done', 'tags_details': [{'tag_id_int': 23, 'tag_rank': 1}, {'tag_id_int': 22, 'tag_rank': 2}, {'tag_id_int': 231, 'tag_rank': 3}, {'tag_id_int': 233, 'tag_rank': 1}, {'tag_id_int': 236, 'tag_rank': 2}], 'unknown': []}
854
{'state': 'done', 'tags_details': [{'tag_id_int': 27, 'tag_rank': 1}, {'tag_id_int': 37, 'tag_rank': 2}, {'tag_id_int': 31, 'tag_rank': 3}, {'tag_id_int': 231, 'tag_rank': 1}, {'tag_id_int': 233, 'tag_rank': 2}, {'tag_id_int': 236, 'tag_rank': 3}], 'unknown': []}
848
{'state': 'done', 'tags_details': [{'tag_id_int': 27, 'tag_rank': 1}, {'tag_id_int': 31, 'tag_rank': 2}, {'tag_id_int': 37, 'tag_rank': 3}, {'tag_id_int': 231, 'tag_rank': 1}, {'tag_id_int': 233, 'tag_rank': 2}, {'tag_id_int': 

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [2]:
conn.close()

In [47]:
from src.server.models.dms_constants import DmsConstants
from src.server.models.dms_model import DmsCache
from src.db.mysql import DbConnection
import re
conn = DbConnection()
prompt_sheet = DmsCache.get_json(conn, DmsConstants.prompts_guid)
words = []
tags = []
tags_not_found = []
for row in prompt_sheet["Genres"]:
    arr = re.findall(r'[a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*', row["Tag"])
    words += arr
    tags.append(row["Tag"])
    if len(arr) == 0:
        print(f"Zero: {row}")
conn.close()
not_found = []
offset = 0
for i in range(len(words)):
    index = i + offset
    if index >= len(words):
        break
    if words[index] == "Hidden":
        print("test")
        words[index] = "xxxx"
    found = False
    potential_word = ""
    for j in range(index, min(index + 5, len(words))):
        potential_word += words[j] + " "
        if potential_word.strip() in tags:
            offset += j - index
            found = True
            break
    if found:
        print(f"Found: {potential_word}")
    else:
        print(f"Not found: {words[index]}")


Found: Hyper-casual 
Found: Casual 
Found: Mid-core 
Found: Hard-core 
Found: Solo 
Found: PvE 
Found: PvP 
Found: Coop 
Found: MMO 
Found: Synchronous 
Found: Asynchronous 
Found: Match 
Found: Puzzle 
Found: Table 
Found: Story 
Found: RPG 
Found: Strategy 
Found: Simulation 
Found: Driving 
Found: Sports 
Found: Combat 
Found: Arcade 
Found: Casino 
Found: Party 
Found: Idle 
Found: Education 
Found: Builder 
Found: Switch 
Found: Blast 
Found: Connect 
Found: Bubble 
Found: Merge 
Found: Brain 
ERROR! Session/line number was not unique in database. History logging moved to new session 2821
test
Not found: xxxx
Not found: Object
Found: Jigsaw 
Found: Physics 
Found: Sudoku 
Found: Word 
Found: Match-shapes 
Found: Crush-blocks 
Found: Escape-room 
Found: Clear path 
Found: Minesweep 
Found: Differences 
Found: Maze 
Found: Nonogram 
Found: Sort water 
Found: Unblock objects 
Found: Control-time 
Found: Travel-time 
Found: Swap-characters 
Found: Backtracking 
Found: Cards 
Found: Co

In [8]:
import re
arr = re.findall(r'[a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*', "Genres: (Primary - Simulation, Secondary - Strategy, Tertiary - Builder), Topics: (Primary - Adventure, Secondary - Reality, Tertiary - Drama)")
arr

['Genres',
 'Primary',
 'Simulation',
 'Secondary',
 'Strategy',
 'Tertiary',
 'Builder',
 'Topics',
 'Primary',
 'Adventure',
 'Secondary',
 'Reality',
 'Tertiary',
 'Drama']